In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stat

## Función para el muestreo

In [ ]:
def muestreoporestrato(df, condicion):
    return df.query(condicion).copy()

#Posible uso:
#df = pd.read_csv('datos.csv')
#condicion = "Variable == 'DViento' & Unidades == 'deg' & `Tiempo de exposición (horas)` == 24"
#estrato = muestreoporestrato(df, condicion)

## Función para resumen inferencial

In [ ]:
import numpy as np
from scipy import stats

def resumen_inferencial(muestra, alpha=0.05):
    """
    Recibe una muestra (array) y devuelve un diccionario con:
    - Estadísticas descriptivas
    - IC para la media (t de Student)
    - IC para la varianza (chi-cuadrado)
    """
    n        = len(muestra)
    media    = np.mean(muestra)
    varianza = np.var(muestra, ddof=1)
    desv_std = np.std(muestra, ddof=1) #desviación estándar muestral 
    gl       = n - 1

    # --- IC para la media (t-Student bilateral) ---
    t_critico   = stats.t.ppf(1 - alpha/2, df=gl)
    margen      = t_critico * (desv_std / np.sqrt(n))
    IC_med_inf  = media - margen
    IC_med_sup  = media + margen

    # --- IC para la varianza (chi-cuadrado bilateral) ---
    chi2_izq    = stats.chi2.ppf(alpha/2,     df=gl)
    chi2_der    = stats.chi2.ppf(1 - alpha/2, df=gl)
    IC_var_inf  = (gl * varianza) / chi2_der
    IC_var_sup  = (gl * varianza) / chi2_izq

    # --- Imprimir resumen ---
    sep = "=" * 45
    print(f"\n  ESTADÍSTICAS DE LA MUESTRA")
    print(sep)
    print(f"  n                    : {n}")
    print(f"  Media muestral (x̄)  : {media:.4f}")
    print(f"  Varianza muestral(S²): {varianza:.4f}")
    print(f"  Desviación std  (S)  : {desv_std:.4f}")

    print(f"\n  INTERVALO DE CONFIANZA PARA LA MEDIA")
    print(f"  t crítico (gl={gl}, α/2): {t_critico:.4f}")
    print(f"  Margen de error       : ± {margen:.4f}")
    print(f"  IC {int((1-alpha)*100)}%: [ {IC_med_inf:.4f} , {IC_med_sup:.4f} ]")

    print(f"\n  INTERVALO DE CONFIANZA PARA LA VARIANZA")
    print(f"  χ² izquierdo (α/2)   : {chi2_izq:.4f}")
    print(f"  χ² derecho   (1-α/2) : {chi2_der:.4f}")
    print(f"  IC {int((1-alpha)*100)}%: [ {IC_var_inf:.4f} , {IC_var_sup:.4f} ]")

    # --- Retornar todo en un diccionario ---
    return {
        "n": n, "media": media, "varianza": varianza, "desv_std": desv_std,
        "IC_media": (IC_med_inf, IC_med_sup),
        "IC_varianza": (IC_var_inf, IC_var_sup)
    }


## Funcion ley de los grandes números y del limite central

In [ ]:
def TLCyLGN(cs_estrato,variable,n,num_muestras):
    N = cs_estrato.shape[0]
    muestra = cs_estrato[variable].dropna().sample(n=n, random_state=42)
    tamaños = list(range(1, N, 10))
    medias_muestrales = []
    media_poblacional = cs_estrato[variable].dropna().mean()
    medias = []
    #LGN
    for n in tamaños:
        muestra = cs_estrato[variable].dropna().sample(n=n, random_state=42)
        medias_muestrales.append(muestra.mean())
    plt.figure(figsize=(8,5))
    plt.plot(tamaños, medias_muestrales, 'bo-', linewidth=2, markersize=8)
    plt.axhline(media_poblacional, color='red', linestyle='--', label=f'Media real: {media_poblacional:.2f}')
    plt.xlabel('Tamaño muestra (n)'); plt.ylabel('Media muestral'); plt.legend(); plt.grid(True); plt.title('Ley de los Grandes Números')
    plt.show()
    #TLC
    for i in range(num_muestras):
        muestra = cs_estrato[variable].dropna().sample(n=n, replace=False)
        medias.append(muestra.mean())
    plt.hist(medias, bins=30)
    plt.xlabel("Media muestral")
    plt.ylabel("Frecuencia")
    plt.title("Teorema del Límite Central")
    plt.show()

#Posible uso:
#TLCyLGN(estrato, 'Valor', 30, 1000)

## Calcular la varianza empirica de una cantida de muestras




In [ ]:
def varemp (mu,sigma,n,m): #mu: media poblacional, sigma: desviación estándar poblacional, n: tamaño de la muestra, m: número de muestras
    np.random.seed(42) 
    medias=[]
    medianas=[]
    for i in range(m):
        muestra=np.random.normal(mu,sigma,n)
        medias.append(np.mean(muestra))
        medianas.append(np.median(muestra))
        
    medias=np.array(medias)
    medianas=np.array(medianas)
    var_media=np.var(medias,ddof=1)
    var_mediana=np.var(medianas,ddof=1) ##ddof=1 para varianza muestral
    return var_media,var_mediana

#Posible uso:
#var_media,var_mediana = varemp(0,1,30,1000)

## Prueba de hipotesis z

In [ ]:
def prueba_hipotesis_completa(x_barra, mu_0, sigma, n, alpha, cola, x_sup=None):
    
    z = (x_barra - mu_0) / (sigma / np.sqrt(n))
    x = np.linspace(-5, 5, 1000)
    y_H0 = stat.norm.pdf(x, 0, 1)
    
    plt.figure(figsize=(10,6))
    plt.plot(x, y_H0, label='Distribución bajo H0')

    # Solo grafica H1 y calcula delta si se proporcionó x_sup
    if x_sup is not None:
        delta = abs(x_sup - mu_0) / (sigma / np.sqrt(n))
        y_H1  = stat.norm.pdf(x, delta, 1)
        plt.plot(x, y_H1, linestyle='--', label='Distribución bajo H1')
    
    match cola:
        case 'inf':
            z_alpha   = stat.norm.ppf(1 - alpha)
            z_crit    = -z_alpha
            decision  = z < z_crit
            p_value   = stat.norm.cdf(z)
            limite_sup = x_barra + z_alpha * (sigma / np.sqrt(n))
            IC        = (-np.inf, limite_sup)
            
            # Beta solo si se conoce x_sup
            beta = stat.norm.cdf(z_alpha + delta) if x_sup is not None else None
            
            plt.fill_between(x, y_H0, where=(x < z_crit), alpha=0.3)
            if x_sup is not None:
                plt.fill_between(x, y_H1, where=(x >= z_crit), alpha=0.3)

        case 'sup':
            z_alpha   = stat.norm.ppf(1 - alpha)
            z_crit    = +z_alpha
            decision  = z > z_crit
            p_value   = 1 - stat.norm.cdf(z)
            limite_inf = x_barra - z_alpha * (sigma / np.sqrt(n))
            IC        = (limite_inf, np.inf)
            
            beta = stat.norm.cdf(z_alpha - delta) if x_sup is not None else None
            
            plt.fill_between(x, y_H0, where=(x > z_crit), alpha=0.3)
            if x_sup is not None:
                plt.fill_between(x, y_H1, where=(x <= z_crit), alpha=0.3)

        case 'dos':
            z_alpha   = stat.norm.ppf(1 - alpha/2)
            z_crit    = z_alpha
            decision  = abs(z) > z_crit
            p_value   = 2 * (1 - stat.norm.cdf(abs(z)))
            limite_inf = x_barra - z_alpha * (sigma / np.sqrt(n))
            limite_sup = x_barra + z_alpha * (sigma / np.sqrt(n))
            IC        = (limite_inf, limite_sup)
            
            beta = (stat.norm.cdf(z_alpha - delta) - 
                    stat.norm.cdf(-z_alpha - delta)) if x_sup is not None else None
            
            plt.fill_between(x, y_H0, where=(x > z_crit),  alpha=0.3)
            plt.fill_between(x, y_H0, where=(x < -z_crit), alpha=0.3)
            if x_sup is not None:
                plt.fill_between(x, y_H1, where=(abs(x) <= z_crit), alpha=0.3)

        case _:
            raise ValueError("cola no válida")

    plt.axvline(z, linestyle=':', label='Z observado')
    plt.title("Prueba de hipótesis completa")
    plt.legend()
    plt.grid()
    plt.show()

    return {
        'z': z,
        'decision': decision,
        'p_value': p_value,
        'beta': beta,   # None si no se pasó x_sup
        'IC': IC
    }


#posible uso:
# Sin x_sup → beta retorna None, no falla
#prueba_hipotesis_completa(94, 100, 15, 25, 0.05, 'inf')

# Con x_sup → calcula beta normalmente
#prueba_hipotesis_completa(94, 100, 15, 25, 0.05, 'inf', x_sup=90)



## Prueba de hiptesis t  (desv poblacional desconocida)

In [ ]:
def prueba_t_media(x_barra, mu_0, s, n, alpha, cola, x_sup=None):
    
    gl = n - 1  # grados de libertad
    t  = (x_barra - mu_0) / (s / np.sqrt(n))
    
    x    = np.linspace(-5, 5, 1000)
    y_H0 = stat.t.pdf(x, df=gl)
    
    plt.figure(figsize=(10, 6))
    plt.plot(x, y_H0, label='Distribución bajo H0')

    if x_sup is not None:
        delta = abs(x_sup - mu_0) / (s / np.sqrt(n))
        y_H1  = stat.t.pdf(x - delta, df=gl)
        plt.plot(x, y_H1, linestyle='--', label='Distribución bajo H1')

    match cola:
        case 'inf':
            t_alpha   = stat.t.ppf(1 - alpha, df=gl)
            t_crit    = -t_alpha
            decision  = t < t_crit
            p_value   = stat.t.cdf(t, df=gl)
            limite_sup = x_barra + t_alpha * (s / np.sqrt(n))
            IC        = (-np.inf, limite_sup)
            beta      = stat.t.cdf(t_alpha + delta, df=gl) if x_sup is not None else None

            plt.fill_between(x, y_H0, where=(x < t_crit), alpha=0.3)
            if x_sup is not None:
                plt.fill_between(x, y_H1, where=(x >= t_crit), alpha=0.3)

        case 'sup':
            t_alpha   = stat.t.ppf(1 - alpha, df=gl)
            t_crit    = +t_alpha
            decision  = t > t_crit
            p_value   = 1 - stat.t.cdf(t, df=gl)
            limite_inf = x_barra - t_alpha * (s / np.sqrt(n))
            IC        = (limite_inf, np.inf)
            beta      = stat.t.cdf(t_alpha - delta, df=gl) if x_sup is not None else None

            plt.fill_between(x, y_H0, where=(x > t_crit), alpha=0.3)
            if x_sup is not None:
                plt.fill_between(x, y_H1, where=(x <= t_crit), alpha=0.3)

        case 'dos':
            t_alpha   = stat.t.ppf(1 - alpha/2, df=gl)
            t_crit    = t_alpha
            decision  = abs(t) > t_crit
            p_value   = 2 * (1 - stat.t.cdf(abs(t), df=gl))
            limite_inf = x_barra - t_alpha * (s / np.sqrt(n))
            limite_sup = x_barra + t_alpha * (s / np.sqrt(n))
            IC        = (limite_inf, limite_sup)
            beta      = (stat.t.cdf(t_alpha - delta, df=gl) - stat.t.cdf(-t_alpha - delta, df=gl)) if x_sup is not None else None

            plt.fill_between(x, y_H0, where=(x >  t_crit), alpha=0.3)
            plt.fill_between(x, y_H0, where=(x < -t_crit), alpha=0.3)
            if x_sup is not None:
                plt.fill_between(x, y_H1, where=(abs(x) <= t_crit), alpha=0.3)

        case _:
            raise ValueError("cola no válida")

    plt.axvline(t, linestyle=':', label='t observado')
    plt.title("Prueba t para la media (σ desconocida)")
    plt.legend()
    plt.grid()
    plt.show()

    return {
        't': t,
        'gl': gl,
        'decision': decision,
        'p_value': p_value,
        'beta': beta,
        'IC': IC
    }
#posible uso:
# Sin x_sup → beta retorna None, no falla
#prueba_t_media(94, 100, 15, 25, 0.05, 'inf')
# Con x_sup → calcula beta normalmente
#prueba_t_media(94, 100, 15, 25, 0.05, 'inf', x_sup=90)
